# Resultados de la etapa 2: de la segmentacion al diagnostico (ACDC)

Este notebook **no entrena nada**: lee lo que dejaron los scripts en `outputs/acdc/` y lo
muestra paso a paso. Para generarlo desde cero (ver `README.md`):

```
uv run python acdc_data.py
for k in 0 1 2 3 4; do uv run python train_acdc.py --fold $k; done
uv run python inferir_acdc.py
uv run python diagnostico.py
uv run python seguridad.py
uv run python figuras_acdc.py
```

Recorrido:
1. Los datos: que distingue a cada grupo
2. Entrenamiento de la U-Net (5 pliegues)
3. Segmentacion: Dice por estructura
4. Mirar un paciente corte a corte
5. Biomarcadores: prediccion contra mascara real
6. Diagnostico
7. Modulo de seguridad

Convencion: **training** = 100 pacientes evaluados *fuera de pliegue* (cada uno lo segmenta
el modelo que no lo vio); **test** = los 50 pacientes oficiales, segmentados por el promedio
de los 5 modelos y nunca usados para decidir nada.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.insert(0, ".")
import acdc_data as A
from figuras_acdc import AZUL, NARANJA, COLOR_CLASE, TINTA, TINTA_2, _estilo

OUT = Path("outputs/acdc")
FIGS = OUT / "figuras"
necesarios = ["pacientes.csv", "diagnostico.json", "seguridad.json", "seguridad_pacientes.csv"]
faltan = [f for f in necesarios if not (OUT / f).exists()]
assert not faltan, f"Faltan {faltan} en {OUT}: corre primero los scripts de la etapa 2"

df = pd.read_csv(OUT / "pacientes.csv")
dx = json.loads((OUT / "diagnostico.json").read_text())
seg = json.loads((OUT / "seguridad.json").read_text())
alertas = pd.read_csv(OUT / "seguridad_pacientes.csv")
tr, te = df[df.split == "training"], df[df.split == "testing"]
pd.set_option("display.precision", 3)
print(f"{len(df)} pacientes: {len(tr)} training (fuera de pliegue) + {len(te)} test")

## 1. Los datos: que distingue a cada grupo

Cada paciente del ACDC pertenece a uno de 5 grupos (30 por grupo):

| grupo | que es | que se espera ver |
|---|---|---|
| NOR | corazon normal | FE del VI ~60 %, volumenes normales |
| MINF | infarto previo | FE del VI baja, pared adelgazada en una region |
| DCM | miocardiopatia dilatada | VI muy grande y FE muy baja |
| HCM | miocardiopatia hipertrofica | pared gruesa, masa alta, FE normal o alta |
| RV | ventriculo derecho anormal | VD dilatado, FE del VD baja |

La tabla usa las **mascaras de los expertos** (columnas `real_`), asi que muestra lo que
el clasificador *podria* aprender si la segmentacion fuera perfecta.

In [ ]:
columnas = {
    "real_vi_fd": "VI FD (ml)", "real_fe_vi": "FE VI (%)",
    "real_vd_fd": "VD FD (ml)", "real_fe_vd": "FE VD (%)",
    "real_masa": "masa (g)", "real_grosor_max": "grosor max (mm)",
    "real_engrosamiento": "engrosamiento (%)",
}
(df.groupby("grupo")[list(columnas)].median().rename(columns=columnas)
   .reindex(list(A.GRUPOS)).round(1))

## 2. Entrenamiento de la U-Net (5 pliegues)

Cada pliegue entreno 100 epocas sobre 80 pacientes y monitoreo el Dice sobre los 20 que
dejo fuera. El monitoreo **no** se uso para elegir la epoca: se guarda siempre la ultima,
porque esas mismas predicciones alimentan despues al clasificador.

Las lineas finas son los 5 pliegues y la gruesa su promedio. Que las 5 curvas casi se
superpongan indica que el resultado no depende de que pacientes tocaron en cada pliegue.

In [ ]:
historias = [json.loads((OUT / f"fold{k}" / "history.json").read_text()) for k in range(5)]
clases = [("VD", "Ventriculo derecho"), ("miocardio", "Miocardio"), ("VI", "Ventriculo izquierdo")]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for h in historias:
    axes[0].plot([f["epoca"] for f in h], [f["train_loss"] for f in h], color=AZUL, alpha=0.35, linewidth=1)
axes[0].set_title("Perdida de entrenamiento (CE + Dice)", fontsize=10, color=TINTA, loc="left")
axes[0].set_xlabel("epoca", fontsize=9, color=TINTA_2)

for ax, (clave, titulo) in zip(axes[1:], clases):
    curvas = []
    for h in historias:
        puntos = [(f["epoca"], f[f"val_dice_{clave}"]) for f in h if f"val_dice_{clave}" in f]
        x, y = zip(*puntos)
        curvas.append(y)
        ax.plot(x, y, color=AZUL, alpha=0.35, linewidth=1)
    media = np.mean(curvas, axis=0)
    ax.plot(x, media, color=AZUL, linewidth=2.5, label="promedio")
    ax.text(x[-1], media[-1], f" {media[-1]:.3f}", fontsize=9, color=TINTA, va="center")
    ax.set_title(f"Dice en el pliegue retenido: {titulo}", fontsize=10, color=TINTA, loc="left")
    ax.set_xlabel("epoca", fontsize=9, color=TINTA_2)
    ax.set_ylim(0.7, 1.0)
for ax in axes:
    _estilo(ax)
plt.tight_layout()
plt.show()

## 3. Segmentacion: Dice por estructura

Dice 3D por paciente (volumen completo, estandar del ACDC), promediado sobre pacientes, en
fin de diastole (FD, corazon lleno) y fin de sistole (FS, contraido). La ultima fila es la
metrica del enunciado del curso: IoU 2D del miocardio, promedio por corte.

Test sale un poco mejor que training porque en test se promedian los 5 modelos (ensamble)
y en training cada paciente lo segmenta un solo modelo.

In [ ]:
filas = []
for nombre, clave in (("VI", "vi"), ("miocardio", "mio"), ("VD", "vd")):
    for split, d in (("training", tr), ("test", te)):
        filas.append({"estructura": nombre, "conjunto": split,
                      "Dice FD": d[f"dice_{clave}_ED"].mean(), "Dice FS": d[f"dice_{clave}_ES"].mean()})
for split, d in (("training", tr), ("test", te)):
    filas.append({"estructura": "IoU 2D miocardio (curso)", "conjunto": split,
                  "Dice FD": d.iou2d_mio_ED.mean(), "Dice FS": d.iou2d_mio_ES.mean()})
pd.DataFrame(filas).set_index(["estructura", "conjunto"]).round(3)

## 4. Mirar un paciente corte a corte

`mostrar_paciente` dibuja todos los cortes de un paciente: la resonancia, la mascara de los
expertos y la prediccion. Colores: azul = VD, naranja = miocardio, aqua = VI.

Ejemplo: `patient135`, un infarto (MINF) que el sistema diagnostico como dilatada (DCM) y que
el modulo de seguridad marco. Cambia el `pid` para ver otro paciente (101 a 150 son test).

In [ ]:
def mostrar_paciente(pid, fase="ED"):
    p = next(x for x in A.listar_pacientes() if x.pid == pid)
    img, gt, _ = A.cargar_fase(p, fase)
    pred = np.load(OUT / "mascaras" / f"{pid}.npz")[fase]
    cortes = [z for z in range(img.shape[-1]) if (gt[..., z] > 0).any() or (pred[z] > 0).any()]
    fig, axes = plt.subplots(3, len(cortes), figsize=(1.9 * len(cortes), 6), squeeze=False)
    for col, z in enumerate(cortes):
        for fila, mascara in enumerate((None, gt[..., z], pred[z])):
            ax = axes[fila, col]
            ax.imshow(img[..., z], cmap="gray")
            if mascara is not None:
                capa = np.zeros((*mascara.shape, 4))
                for c, color in COLOR_CLASE.items():
                    capa[mascara == c] = (*mcolors.to_rgb(color), 0.55)
                ax.imshow(capa)
            ax.set_xticks([]); ax.set_yticks([])
        axes[0, col].set_title(f"corte {z}", fontsize=8, color=TINTA_2)
    for fila, nombre in enumerate(("resonancia", "real", "prediccion")):
        axes[fila, 0].set_ylabel(nombre, fontsize=9, color=TINTA)
    r = df[df.pid == pid].iloc[0]
    a = alertas[alertas.pid == pid].iloc[0]
    fig.suptitle(f"{pid} ({fase}) | real {r.grupo}, diagnostico {a.pred} | "
                 f"FE VI pred {r.pred_fe_vi:.1f} % vs real {r.real_fe_vi:.1f} % | "
                 f"{'ALERTA' if a.alerta else 'sin alerta'}", x=0.01, ha="left", fontsize=10)
    plt.tight_layout()
    plt.show()

mostrar_paciente("patient135")

## 5. Biomarcadores: prediccion contra mascara real

Para cada paciente de test se compara el biomarcador medido sobre la mascara predicha con el
medido sobre la mascara de los expertos. **Sesgo** = error promedio (con signo), **DE** =
dispersion del error, **r** = correlacion. En el grafico de Bland-Altman cada punto es un
paciente; las lineas punteadas son los limites de acuerdo (sesgo +- 1.96 DE): ahi cae el 95 %
de los errores.

In [ ]:
nombres = {"fe_vi": "FE VI (puntos %)", "fe_vd": "FE VD (puntos %)", "vi_fd": "VI FD (ml)",
           "vd_fd": "VD FD (ml)", "masa": "masa (g)", "grosor_max": "grosor max (mm)"}
filas = []
for n, etiqueta in nombres.items():
    err = te[f"pred_{n}"] - te[f"real_{n}"]
    filas.append({"biomarcador": etiqueta, "sesgo": err.mean(), "DE": err.std(),
                  "r": np.corrcoef(te[f"pred_{n}"], te[f"real_{n}"])[0, 1]})
display(pd.DataFrame(filas).set_index("biomarcador").round(3))
Image(str(FIGS / "bland_altman.png"))

## 6. Diagnostico

El clasificador (regresion logistica, elegida por validacion cruzada en training con una regla
fijada de antemano) recibe los 16 biomarcadores de cada paciente.

- **Pipeline**: biomarcadores de las mascaras predichas. Es el sistema real.
- **Oraculo**: biomarcadores de las mascaras de los expertos. Es la cota superior: lo mejor que
  podria hacer este clasificador si la segmentacion fuera perfecta.

In [ ]:
for clave in ("pipeline", "oraculo"):
    r = dx[clave]
    print(f"{clave:9s} exactitud test {r['exactitud_test']:.0%} | CV training "
          + ", ".join(f"{k} {v:.1%}" for k, v in r["cv_training"].items()) + f" -> {r['elegido']}")

probs = pd.read_csv(OUT / "diagnostico_pacientes.csv")
errores = probs[(probs.split == "testing") & (probs.pred != probs.grupo)]
print("\nPacientes de test mal diagnosticados:")
display(errores.drop(columns="split").set_index("pid").round(2))
Image(str(FIGS / "confusion.png"))

## 7. Modulo de seguridad

Cada paciente recibe dos medidas de duda, calculables **sin conocer la respuesta**:

1. **Duda de segmentacion**: cuanto cambian los biomarcadores entre las 8 vistas del TTA
   (la misma imagen rotada y volteada). Si la red da respuestas distintas segun la
   orientacion, no esta segura.
2. **Margen del clasificador**: probabilidad del diagnostico elegido menos la del segundo.
   Margen chico = dos diagnosticos casi empatados.

Se marca un paciente si la duda de segmentacion supera el percentil 90 de training o si el
margen queda bajo el percentil 10 de training. Los umbrales se fijaron con training y se
aplicaron sin cambios a test.

In [ ]:
for split in ("training", "testing"):
    s = seg[split]
    print(f"{split:8s}: marcados {s['marcados']}/{s['n']} | errores atrapados {s['errores_atrapados']} | "
          f"exactitud no marcados {s['exactitud_no_marcados']:.1%} (todos {s['exactitud_todos']:.1%})")

print("\nPacientes de test marcados o mal diagnosticados:")
cols = ["pid", "grupo", "pred", "acierto", "margen", "duda_seg", "alerta_seg", "alerta_dx"]
display(alertas[(alertas.split == "testing") & (alertas.alerta | ~alertas.acierto)][cols]
        .set_index("pid").round(2))
Image(str(FIGS / "riesgo_cobertura.png"))

La curva riesgo-cobertura ordena a los pacientes de menor a mayor duda y muestra la exactitud
si solo se aceptaran automaticamente los primeros. Con todos los pacientes (100 %) es la
exactitud del sistema; al dejar fuera a los mas dudosos, la exactitud de los que quedan sube.
En la practica: el sistema podria informar automaticamente los casos seguros y mandar los
dudosos a un cardiologo.

**Limites**: con 50 pacientes de test, cada error mueve la exactitud en 2 puntos; y la
correlacion entre duda de segmentacion y error real es moderada (Spearman ~0.2-0.4). La duda
sirve para ordenar que revisar primero, no como garantia.

In [ ]:
Image(str(FIGS / "segmentaciones_test.png"))